# SPIDER reviewer: QLoRA fine-tune on a free Colab T4

Fine-tunes the Llama 3.2 3B base on the labeled samples in `dataset/` so that the model answers in the exact JSON the SPIDER scanner expects, then exports a GGUF for Ollama.
Defensive security review only: the training targets contain findings, explanations and fixed code, never exploits.

**Before you start:** Runtime > Change runtime type > **T4 GPU**. Run the cells top to bottom (about 30-90 min in total; the training cell dominates, timings are estimates, not measured).
Nothing here needs a Hugging Face token: the default base `chuanli11/Llama-3.2-3B-Instruct-uncensored` is ungated.

In [ ]:
import torch, subprocess
assert torch.cuda.is_available(), "No GPU: Runtime > Change runtime type > T4 GPU"
print(torch.cuda.get_device_name(0), round(torch.cuda.get_device_properties(0).total_memory/1e9,1), "GB")

In [ ]:
!pip install -q -U "transformers>=4.46" peft accelerate bitsandbytes sentencepiece gguf protobuf
!git clone -q https://github.com/Mahdy0000/SPIDER.git
%cd SPIDER

## 1. Build the instruction-tuning data
Uses the real SPIDER prompts, so the model learns the tool's own format. Held-out validation rows are never trained on.

In [ ]:
!python training/prep_data.py | tail -40

## 2. Train (QLoRA, 4-bit NF4, LoRA r=16 on all linear layers)
Defaults: 3 epochs, lr 2e-4, 2048-token window, effective batch 16. If you hit CUDA out-of-memory, add `--max-seq 1536` or `--lora-r 8`.

In [ ]:
!python training/train_qlora.py --out training/out/adapter

## 3. Merge the adapter into the base model (CPU, fp16)

In [ ]:
!python training/merge_export.py --adapter training/out/adapter --out training/out/merged

## 4. Convert to GGUF and quantize
`Q4_K_M` (about 2 GB) is the default. Use `Q8_0` (about 3.4 GB) if you have the memory and want the closest match to the fine-tuned weights. Building `llama-quantize` takes a few minutes on Colab's CPUs.

In [ ]:
!SKIP_OLLAMA=1 bash training/convert_to_ollama.sh training/out/merged Q4_K_M
!ls -lh training/out/*.gguf training/out/Modelfile

## 5. Download the result
Saves the GGUF and Modelfile to Google Drive (recommended, the file is about 2 GB) and keeps the training log.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/spider-reviewer
!cp training/out/spider-reviewer.Q4_K_M.gguf training/out/Modelfile training/out/adapter/train_log.json /content/drive/MyDrive/spider-reviewer/
!ls -lh /content/drive/MyDrive/spider-reviewer

## 6. On your own machine
Put the two files in one folder, then:
```bash
ollama create spider-reviewer -f Modelfile
```
SPIDER then picks `spider-reviewer` automatically (override any time with `--model` or `SPIDER_MODEL`).
Measure before/after with `training/evaluate.py` as described in the README ("Fine-tuning" section). The session's files vanish when Colab disconnects, so copy the outputs first.